In [1]:
import anndata as ad
import pathlib
from typing import Union

def load_h5ad_data(file_path: Union[str, pathlib.Path]) -> ad.AnnData:
    """
    Reads an .h5ad file and returns an AnnData object.

    Parameters:
    -----------
    file_path : str or Path
        The path to the .h5ad file you want to load.

    Returns:
    --------
    data : ad.AnnData
        The loaded AnnData object.

    Raises:
    -------
    FileNotFoundError:
        If the file does not exist at the specified path.
    """
    path = pathlib.Path(file_path)
    
    if not path.exists():
        raise FileNotFoundError(f"No file found at: {path.absolute()}")
    
    if path.suffix != ".h5ad":
        print(f"Warning: The file extension {path.suffix} is not .h5ad.")

    try:
        data = ad.read_h5ad(path)
        print(f"Successfully loaded data: {data.n_obs} observations and {data.n_vars} variables.")
        return data
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
        raise

In [3]:
adata = load_h5ad_data(file_path="data/SNB_EXP_1054/counts.h5ad")

Successfully loaded data: 27181 observations and 6059 variables.


In [13]:
import pandas as pd
import pathlib
from typing import Union

def load_cell_metadata(file_path: Union[str, pathlib.Path], sep: str = ",") -> pd.DataFrame:
    """
    Reads a CSV file and returns a DataFrame containing only 'cellID' and 'celltype'.

    Parameters:
    -----------
    file_path : str or Path
        Path to the metadata CSV file.

    Returns:
    --------
    df : pd.DataFrame
        A DataFrame with exactly two columns: ['cellID', 'celltype'].

    Raises:
    -------
    FileNotFoundError:
        If the file does not exist.
    ValueError:
        If the required columns 'cellID' or 'celltype' are missing.
    """
    path = pathlib.Path(file_path)
    
    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path.absolute()}")

    # Read the CSV
    df = pd.read_csv(path, sep=sep, index_col=False)

    # Define required columns
    required_cols = {"cellID", "celltype"}
    
    # Check if columns exist (case-sensitive)
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns in {path.name}: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    # Return only the subset of columns
    return df[list(required_cols)].copy()

In [ ]:
metadata = load_cell_metadata(file_path="data/SNB_EXP_1054/metadata_clean.csv", sep  = ";")

/tmp/ipykernel_4291/292181773.py:32: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(path, sep=sep, index_col=False)


array(['ID01_Veh_TG_S1_TGTCACCCAGTTGAGT-1',
       'ID01_Veh_TG_S1_CACCATTTCTGTGCAA-1',
       'ID01_Veh_TG_S1_CGTATGATCGAACGCT-1', ...,
       'ID14_cNTG_TG_S4_TACACAACAAGTAGCT-1',
       'ID14_cNTG_TG_S4_CAAAGCGAGTGCCTGG-1',
       'ID14_cNTG_TG_S4_TCTACTTGTCAGTGCC-1'], shape=(27181,), dtype=object)

In [15]:
from typing import Tuple, Optional

def h5ad_metadata_checker(h5ad_object: ad.AnnData, metadata: pd.DataFrame):
    """
    Loads h5ad and CSV files, then validates if the cells match.
    
    Returns:
        ad.AnnData: The AnnData object with metadata attached if successful.
        None: If validation fails.
    """

    # Get IDs from both
    h5ad_ids = set(h5ad_object.obs_names)
    csv_ids = set(metadata['cellID'])
    
    n_h5ad = len(h5ad_ids)
    n_csv = len(csv_ids)

    # Calculate matches
    matches = h5ad_ids.intersection(csv_ids)
    n_matches = len(matches)

    # CASE 1: Perfect Match
    if n_h5ad == n_csv and n_matches == n_h5ad:
        print("SUCCESS: Dimensions and IDs match perfectly.")
        # Attach metadata to AnnData for convenience
        adata.obs = metadata.set_index('cellID').loc[adata.obs_names]
        return True

    # CASE 2: Size mismatch, but some IDs match
    print("❌ ERROR: Mismatch detected.")
    print(f"- Cells in H5AD: {n_h5ad}")
    print(f"- Cells in CSV: {n_csv}")
    
    if n_matches == 0:
        print("- Result: No cellIDs match between files.")
    else:
        print(f"- Result: Only {n_matches} out of {n_h5ad} cellIDs match.")
        
    return False

In [18]:
valid_files = h5ad_metadata_checker(h5ad_object = adata, metadata = metadata)

SUCCESS: Dimensions and IDs match perfectly.


In [22]:
import numpy as np

def split_data(
    adata: ad.AnnData, 
    metadata: pd.DataFrame, 
    train_size: float = 0.65, 
    seed: int = 42
) -> Tuple[ad.AnnData, ad.AnnData, pd.DataFrame, pd.DataFrame]:
    """
    Splits AnnData and metadata into training and testing sets.

    Parameters:
    -----------
    adata : ad.AnnData
        The high-dimensional gene expression data.
    metadata : pd.DataFrame
        The metadata containing 'cellID' and 'celltype'.
    train_size : float
        Proportion of data for training (default 0.65).
    seed : int
        Random seed for reproducibility.

    Returns:
    --------
    train_adata, test_adata, train_meta, test_meta
    """
    # 1. Set the seed for reproducibility
    np.random.seed(seed)

    # 2. Ensure IDs are aligned
    # We use the intersection to be safe
    common_ids = list(set(adata.obs_names).intersection(set(metadata['cellID'])))
    
    if len(common_ids) == 0:
        raise ValueError("No common cellIDs found between h5ad and metadata.")

    # 3. Shuffle the IDs
    shuffled_ids = np.array(common_ids)
    np.random.shuffle(shuffled_ids)

    # 4. Calculate split point
    split_idx = int(len(shuffled_ids) * train_size)
    train_ids = shuffled_ids[:split_idx]
    test_ids = shuffled_ids[split_idx:]

    # 5. Perform the split
    # AnnData slicing: adata[list_of_obs_names]
    train_adata = adata[train_ids].copy()
    test_adata = adata[test_ids].copy()

    # Metadata slicing
    # We set cellID as index temporarily to make selection easy and fast
    meta_indexed = metadata.set_index('cellID')
    train_meta = meta_indexed.loc[train_ids].reset_index()
    test_meta = meta_indexed.loc[test_ids].reset_index()

    print(f"Data split complete (Seed: {seed})")
    print(f"   Train: {len(train_ids)} cells")
    print(f"   Test:  {len(test_ids)} cells")

    return train_adata, test_adata, train_meta, test_meta

In [26]:
train_adata, test_adata, train_metadata, test_metadata = split_data(adata, metadata, train_size=0.65, seed=42)

Data split complete (Seed: 42)
   Train: 17667 cells
   Test:  9514 cells


In [27]:
train_adata.obs = train_metadata.set_index('cellID').loc[train_adata.obs_names]

In [25]:
def simulate_pseudobulk(
    adata: ad.AnnData,
    n_samples: int = 1000,
    cells_per_sample: int = 500,
    seed: int = 42,
    output_path: str = "data/simulated"
) -> str:
    """
    Simulates artificial bulk RNA-seq samples by aggregating single-cell data.
    
    Parameters:
    -----------
    adata : ad.AnnData
        The input scRNA-seq data (must have 'celltype' in adata.obs).
    n_samples : int
        Number of bulk samples to generate.
    cells_per_sample : int
        Total cells (N_total) to sum for one bulk profile.
    output_path : str
        Directory to save the compressed results.
    """
    np.random.seed(seed)
    pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)
    
    # Get unique cell types
    cell_types = adata.obs['celltype'].unique()
    n_genes = adata.n_vars
    
    # Prepare storage for results
    bulk_matrix = np.zeros((n_samples, n_genes), dtype=np.float32)
    fractions_list = []

    # Pre-group indices by cell type for speed
    indices_by_type = {
        ct: np.where(adata.obs['celltype'] == ct)[0] 
        for ct in cell_types
    }

    print(f"Generating {n_samples} pseudobulk samples...")

    for i in range(n_samples):
        # 1. Generate random fractions (rc / sum(rc))
        r = np.random.rand(len(cell_types))
        fractions = r / r.sum()
        
        # 2. Calculate Nc (number of cells for each type)
        # We use round to get integers; slight adjustment to ensure it hits cells_per_sample
        n_cells_per_type = np.round(fractions * cells_per_sample).astype(int)
        
        sample_profile = np.zeros(n_genes)
        
        # 3. Sample and Sum
        for idx, ct in enumerate(cell_types):
            count = n_cells_per_type[idx]
            if count > 0:
                # Randomly pick Nc indices for this cell type
                available_indices = indices_by_type[ct]
                # If we need more cells than available, we allow replacement
                chosen_indices = np.random.choice(
                    available_indices, size=count, replace=True
                )
                
                # Sum the expression values
                # .X might be sparse, so we ensure it's handled
                subset = adata.X[chosen_indices]
                if hasattr(subset, "toarray"):
                    subset = subset.toarray()
                
                sample_profile += subset.sum(axis=0)
        
        bulk_matrix[i, :] = sample_profile
        fractions_list.append(fractions)

    # 4. Save efficiently
    fractions_df = pd.DataFrame(fractions_list, columns=cell_types)
    
    # Save as compressed NumPy files
    np.savez_compressed(
        f"{output_path}/pseudobulk_data.npz", 
        X=bulk_matrix, 
        genes=adata.var_names.values
    )
    fractions_df.to_csv(f"{output_path}/fractions.csv", index=False)

    print(f"✅ Simulation complete. Saved to {output_path}")
    return output_path

In [30]:
simulate_pseudobulk(adata=train_adata, n_samples = 10, output_path = "data/simulated")

Generating 10 pseudobulk samples...
✅ Simulation complete. Saved to data/simulated


'data/simulated'